## Tavily Search API

In [2]:
import sys
!{sys.executable} -m pip install tavily-python


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"


In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})
    

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)


{'result': [{'url': 'https://media.jreast.co.jp/tags/9756',
   'title': '東京駅イベントに関する最新情報・おすすめ記事 - JREメディア',
   'content': ':   「東京駅イベント」でタグ付けされた記事一覧です。JREメディアには「東京駅イベント」に関する記事やご案内、便利な情報が8件掲載されています。. # 東京駅で群馬溫泉文化旅遊！温泉王国ぐんま体感イベントを開催. 2026年2月20日（金）～22日（日）に、JR東日本東京駅B1「スクエア ゼロ」で開催される「群馬溫泉文化旅遊 in 東京站」の見どころを詳しく紹介します。 群馬の温泉文化パネル展示や「卯三郎こけし」絵付け体験、ぐんまちゃんグリーティング、カラフルだるまや名産品販売、温泉宿泊券（補助券）等が当たる抽選会まで、東京駅で群馬の温泉旅気分を味わえるポイントを分かりやすく解説します。. # 東京駅で開催！青森・北海道道南産直市＠スクエアゼロ. 「青森県・函館観光キャンペーン」期間中、JR東日本クロスステーションは、エキナカ地域フェア「青森・北海道道南MEGURIP（めぐりっぷ）」のエキナカイベントとして、2026年1月14日～18日まで「青森・北海道道南産直市」を東京駅イベントスペース「スクエア ゼロ」で開催します。青森・北海道道南エリアの特産品や農産物、加工品、お菓子など美味しい地産品が集結。青森・北海道道南エリアの魅力を発信します。. # 東京駅で開催中の春イベント「TOKYO EASTER & SWEETS」をレポート！. 2025年4月20日(日)まで東京駅特設会場で開催中の「TOKYO EASTER&SWEETS」。対象店舗でお買い物(金額制限なし)をすると、イースターエッグにちなんだアトラクションに参加でき、獲得したポイントに応じてオリジナルチョコレート(※数量限定)もGET! # 東京駅発春イベント「TOKYO EASTER & SWEETS」開催！. JR東京駅B1改札内イベントスペース「スクエア ゼロ」にて、春の訪れを祝福するお祭り”イースター”をテーマにした「TOKYO EASTER & SWEETS」を2025年4月12日(土)～20日(日)開催！ 特設会場では、春をイメージしたスイーツやフラワ

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較します。

- **東京都の面積**: 約2,194平方キロメートル
- **沖縄県の面積**: 約2,271平方キロメートル

これにより、沖縄県の方が広いということになります。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内の東京駅周辺でのイベント情報をいくつかご紹介します。

1. **[これからの1カ月のイベント予定](https://www.instagram.com/p/DP09A1gkUAo/)**  
   - 日時: 10月23日（木）  
   - 内容: 東京駅・本館Bにて、素敵なイベントがあるようです。

2. **[東京のイベント一覧 - Enjoy Tokyo](https://www.enjoytokyo.jp/event/list/)**  
   - 内容: 東京駅周辺で進行中の様々なイベントが紹介されています。2026年に向けた長期的なイベントも多数予定されています。

3. **[Time Out Tokyo - 今日は何ができるか](https://www.timeout.jp/tokyo/ja/things-to-do/today)**  
   - 内容: 今日の東京でのアクティビティについて。東京都内で美術館やイベントなどが紹介されています。

4. **[最新PC体験イベント - Impress Watch](https://pc.watch.impress.co.jp/docs/news/2085810.html)**  
   - 内容: 東京駅での最新PCの体験イベントが開催される予定です。

5. **useums Tokyo - アート周遊イベント](https://6museums.tokyo/)**  
   - 内容: 東京駅周辺の美術館を回るアートイベントが開催中。周辺6つの美術館を含むイベントです。

これらのイベント情報は、特に興味のある方にはおすすめの情報です。詳しい内容や日程については、それぞれのリンクをチェックしてみてください。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！どのようにお手伝いできますか？


'質問:東海地方には、どのような県がありますか？'

東海地方には、以下の4つの県があります。

1. 愛知県 (あいちけん)
2. 岐阜県 (ぎふけん)
3. 静岡県 (しずおかけん)
4. 三重県 (みえけん)

これらの県は、日本の中部地方の南部に位置しています。


'質問:愛知県のお土産について検索した結果を教えて'

愛知県のお土産に関する情報を以下にまとめました。

1. **愛知の名産品**  
   - **みそ**: 愛知県は特に味噌が名産で、地元の特産品である「みそ」を使った製品が多いです。特に赤味噌が有名です。

2. **スイーツ系お土産**  
   - **八丁味噌饅頭**: 伝統的な饅頭で、しっとりとした生地に甘いあんこが詰まっています。
   - **抹茶アイスや抹茶のお菓子**: 愛知県茶葉を使用したアイスやお菓子も人気です。

3. **名古屋名物**  
   - **ひつまぶし**: うなぎの料理で、食事として地元民や観光客に人気があります。家庭用のひつまぶしセットも人気です。
   - **手羽先**: 名古屋名物の手羽先も、冷凍食品として販売されることがあります。

4. **趣向を凝らしたお土産**  
   - **味噌カツ**: 味噌カツのタレや、その材料をセットにしたお土産もあります。
   - **名古屋コーチン**: 特選肉が使われた加工品も人気です。

5. **その他の代表的なお土産**  
   - **名古屋のスイーツ**: シュガーバターやドーナツなど、地元のスイーツも観光客に人気があります。

### 参考リンク
- [愛知県のお土産の詳細](https://www.shizuoka-life.jp/post-12561/)
- [愛知の特産お土産と情報](https://aichinow.pref.aichi.jp/souvenirs/)

愛知県のお土産は、伝統的な食材を使用したものから、スイーツや加工食品まで多様性があり、訪れる際にはぜひお土産を楽しんでいただきたいです。

---ご利用ありがとうございました！---
